In [ ]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
df = pd.read_csv(r"/content/spam.csv", encoding='latin-1')

# Keep only needed columns
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [ ]:
# Check missing values
print(df.isnull().sum())

# Remove duplicates
df.drop_duplicates(inplace=True)

# Reset index
df.reset_index(drop=True, inplace=True)

label      0
message    0
dtype: int64


In [ ]:
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()  # lowercase
    text = re.sub('[^a-zA-Z]', ' ', text)  # remove special chars
    words = text.split()

    words = [ps.stem(word) for word in words if word not in stop_words]

    return ' '.join(words)

In [ ]:
df['cleaned_text'] = df['message'].apply(preprocess)

print(df[['message', 'cleaned_text']].head())

                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                        cleaned_text  
0  go jurong point crazi avail bugi n great world...  
1                              ok lar joke wif u oni  
2  free entri wkli comp win fa cup final tkt st m...  
3                u dun say earli hor u c alreadi say  
4               nah think goe usf live around though  


In [ ]:
vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df['cleaned_text'])
print(X.shape)
print(vectorizer.get_feature_names_out()[:10])
print(X.toarray()[0][:10])

(5169, 3000)
['aah' 'abi' 'abiola' 'abj' 'abl' 'abt' 'abta' 'ac' 'acc' 'accept']
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42)
df['cluster'] = kmeans.fit_predict(X)
print(df['cluster'].head())

0    0
1    1
2    1
3    1
4    1
Name: cluster, dtype: int32


In [ ]:
score = silhouette_score(X, df['cluster'])
print("Silhouette Score:", score)

Silhouette Score: 0.004134349838432495


In [ ]:
print(df.groupby('cluster').head(5))

   label                                            message  \
0    ham  Go until jurong point, crazy.. Available only ...   
1    ham                      Ok lar... Joking wif u oni...   
2   spam  Free entry in 2 a wkly comp to win FA Cup fina...   
3    ham  U dun say so early hor... U c already then say...   
4    ham  Nah I don't think he goes to usf, he lives aro...   
5   spam  FreeMsg Hey there darling it's been 3 week's n...   
21   ham  IÛ÷m going to try for 2 months ha ha only joking   
23   ham  Aft i finish my lunch then i go str down lor. ...   
31   ham  Yeah he got in at 2 and was v apologetic. n ha...   
35   ham  Yup... Ok i go home look at the timings then i...   

                                         cleaned_text  cluster  
0   go jurong point crazi avail bugi n great world...        0  
1                               ok lar joke wif u oni        1  
2   free entri wkli comp win fa cup final tkt st m...        1  
3                 u dun say earli hor u c alre